# Cat vs Dog Image Classifier — CNN in Keras/TensorFlow

This notebook builds an image classifier that tells cats and dogs apart, using a
Convolutional Neural Network trained on the Kaggle **Dogs vs. Cats** dataset. It follows the
same workflow as the walkthrough it's based on: download the dataset straight into Colab
with the Kaggle API, load images efficiently with a Keras data pipeline (so the whole
dataset never has to sit in RAM at once), build a CNN with three convolutional blocks,
train it, and use it to predict on new photos.

Every code cell below is followed by a short explanation of what it does and why it's
there — written so you can follow the notebook top to bottom even if some of these Keras
calls are new to you.

**Before you run this:** you'll need a free Kaggle account and an API token
(`kaggle.json`) — [instructions here](https://www.kaggle.com/docs/api#authentication).
This notebook is written for **Google Colab** specifically (the dataset download and file
paths assume a Colab `/content` environment).

In [ ]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================
import os
import zipfile

import numpy as np
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Flatten, Dense, Dropout
)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

**What this cell does.** Pulls in everything the rest of the notebook needs:

- `os`, `zipfile` — standard library tools for handling the downloaded dataset archive.
- `numpy`, `matplotlib.pyplot` — array handling and the accuracy/loss plots later on.
- `cv2` (OpenCV) — used at the very end to load and resize a new photo for prediction.
- `tensorflow` and the specific `keras` pieces — `Sequential` is the model container;
  `Conv2D`, `MaxPooling2D`, `BatchNormalization`, `Flatten`, `Dense`, `Dropout` are the
  layer types the CNN in Section 4 is built from.

The two `print()` lines are just a sanity check — confirming the TensorFlow version and
whether Colab has handed you a GPU. Training on ~25,000 images is *much* faster with one:
in Colab, go to **Runtime → Change runtime type → GPU** before continuing if this prints
an empty list.

In [ ]:
# ==========================================
# 2. DOWNLOAD THE DATASET FROM KAGGLE
# ==========================================
!pip install -q kaggle

from google.colab import files
print("Upload your kaggle.json API token:")
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d salader/dogs-vs-cats

**What this cell does.** Gets the dataset into Colab without you ever downloading it
to your own machine:

1. `pip install -q kaggle` installs the Kaggle command-line tool (`-q` just keeps the
   install output quiet).
2. `files.upload()` opens a file picker in the Colab UI — select your `kaggle.json` token
   here (downloaded from your Kaggle account under *Settings → API → Create New Token*).
3. The next three `!` lines are shell commands (the `!` runs them in the underlying Linux
   shell rather than as Python): they create the hidden `~/.kaggle` folder Kaggle's CLI
   expects, copy your token into it, and lock its file permissions down to `600`
   (owner read/write only) — the Kaggle CLI actually refuses to run if the permissions are
   looser than that, since the file contains your API secret.
4. `kaggle datasets download -d salader/dogs-vs-cats` downloads the dataset itself as a
   zip file into the current Colab session.

**Note:** this cell can't be run or verified in the environment these notes were written
in (no internet access there), so it hasn't been executed end-to-end — it's written from
the standard Kaggle API workflow and should run as-is in Colab once you supply your own
token.

In [ ]:
# ==========================================
# 3. UNZIP THE DATASET
# ==========================================
with zipfile.ZipFile('/content/dogs-vs-cats.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')

print("Train folder contents:", os.listdir('/content/train')[:5])
print("Test folder contents:", os.listdir('/content/test')[:5])

**What this cell does.** Unzips the downloaded archive into `/content`. This
particular dataset extracts into `/content/train/` and `/content/test/`, each already
split into `cats/` and `dogs/` subfolders — that folder structure matters a lot in the
next section, because Keras will read the class labels directly from it. The two
`print()` lines just confirm the extraction worked and show you the first few
files/folders in each directory.

In [ ]:
# ==========================================
# 4. BUILD THE DATA PIPELINE
# ==========================================
IMG_SIZE = (256, 256)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    directory='/content/train',
    labels='inferred',
    label_mode='int',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    directory='/content/test',
    labels='inferred',
    label_mode='int',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

print("Class names (label 0, label 1):", train_ds.class_names)

**What this cell does.** This is the "Keras generator" step from the walkthrough —
`image_dataset_from_directory` builds a `tf.data.Dataset` pipeline that reads images off
disk in batches **as training runs**, rather than loading all ~25,000 images into RAM
up front. A few things worth knowing about the arguments:

- `labels='inferred'` tells Keras to figure out the class of each image from which
  subfolder it's in (`cats/` vs `dogs/`) — you never have to hand-label anything.
- `label_mode='int'` gives each image a single integer label (0 or 1), which is what a
  `sigmoid` output layer with `binary_crossentropy` loss expects (used in Section 5).
- `image_size=(256, 256)` resizes every image to 256×256 as it's loaded — this **must**
  match the `input_shape=(256, 256, 3)` in the first `Conv2D` layer below, or the model
  will fail to build.
- Keras assigns labels **alphabetically** by folder name, so `cats` → label 0 and
  `dogs` → label 1. The final `print()` confirms this via `train_ds.class_names` — keep
  this order in mind for interpreting predictions in Section 8.

`validation_ds` is built the same way from the `test/` folder, and is what `model.fit`
will check the model against after every epoch — images it never trains on.

In [ ]:
# ==========================================
# 5. NORMALIZE PIXEL VALUES
# ==========================================
def process(image, label):
    image = tf.cast(image / 255., tf.float32)
    return image, label

train_ds = train_ds.map(process)
validation_ds = validation_ds.map(process)

**What this cell does.** Raw image pixels are integers from 0 to 255. Neural
networks train far better on small, centered numbers, so this rescales every pixel into
the `[0, 1]` range by dividing by 255 — exactly the normalization step called out in the
walkthrough. `.map(process)` applies that function to every image as it streams through
the pipeline, lazily, one batch at a time — it doesn't rescale the whole dataset up
front, it rescales each batch right when it's needed, which is what keeps this
memory-efficient.

## Model architecture

The CNN below is three convolutional blocks of increasing width (32 → 64 → 128
filters), each followed by batch normalization and max pooling, then two dense layers
with dropout before a single sigmoid output. This is the *improved* version of the
architecture — the walkthrough's first attempt (just the three conv blocks + dense
layers, no `BatchNormalization` or `Dropout`) trained fine but started overfitting:
training accuracy kept climbing while validation accuracy stalled or got worse. Adding
`BatchNormalization` after each conv layer and `Dropout` after each dense layer is what
closed that gap, so they're included here from the start rather than added later.

In [ ]:
# ==========================================
# 4. CNN MODEL ARCHITECTURE
# ==========================================
model = Sequential()

# 1st Convolutional Block
model.add(Conv2D(32, kernel_size=(3, 3), padding='valid', activation='relu', input_shape=(256, 256, 3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

**What this cell does.** The first convolutional block:

- `Conv2D(32, kernel_size=(3,3), padding='valid', activation='relu', input_shape=(256,256,3))`
  slides 32 different 3×3 filters over the 256×256×3 input image. `padding='valid'` means
  *no* padding is added, so each 3×3 filter shrinks the spatial size by 2 pixels per side:
  $256 - 3 + 1 = 254$, giving an output of **254×254×32**. `activation='relu'` zeroes out
  any negative filter response, which is what lets the network learn nonlinear
  combinations of features rather than only linear ones.
- `BatchNormalization()` rescales each of the 32 channels to have roughly zero mean and
  unit variance *within each training batch*, then lets the network learn its own scale
  and shift back on top of that. In practice this stabilizes and speeds up training,
  and — per the walkthrough — it's one of the two changes that fixed this model's
  overfitting.
- `MaxPooling2D(pool_size=(2,2), strides=2, padding='valid')` halves the spatial size by
  keeping only the largest value in each non-overlapping 2×2 window:
  $\lfloor(254-2)/2\rfloor+1 = 127$, giving **127×127×32**. This is both a computational
  saving (a quarter as many values to process next) and a small amount of built-in
  tolerance to a feature shifting by a pixel or two.

In [ ]:
# 2nd Convolutional Block
model.add(Conv2D(64, kernel_size=(3, 3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

**What this cell does.** The same pattern repeated with **64** filters instead of
32 — doubling the filter count is the standard way to grow a CNN's capacity as spatial
size shrinks, since a smaller feature map costs less to convolve even with more filters.
Working through the same formulas: conv shrinks 127→**125** ($127-3+1$), then pooling
halves it to **62×62×64** ($\lfloor(125-2)/2\rfloor+1$). Notice this layer no longer
needs `input_shape` — Keras infers it automatically from whatever the previous layer
output.

In [ ]:
# 3rd Convolutional Block
model.add(Conv2D(128, kernel_size=(3, 3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

**What this cell does.** A third block, now at **128** filters, following the same
32→64→128 doubling pattern. Shapes: conv takes 62→60 ($62-3+1$), pooling takes
60→30, giving a final feature map of **30×30×128**. By this point the network has gone
from "raw pixels" to 128 separate, increasingly abstract feature maps, each summarizing a
30×30 grid of the original 256×256 image.

In [ ]:
# Flattening and Fully Connected Layers
model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.1))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.1))

**What this cell does.**

- `Flatten()` collapses the 30×30×128 feature volume into a single vector of length
  $30 \times 30 \times 128 = 115{,}200$. This is the hinge point between the
  spatially-organized convolutional part of the network and the fully connected part —
  from here on, every unit can mix information from anywhere in the image.
- `Dense(128, activation='relu')` is a fully connected layer that combines all 115,200
  flattened values into 128 new features. This is by far the largest layer in the whole
  network parameter-wise: $115{,}200 \times 128 + 128 = 14{,}745{,}728$ weights, roughly
  **99% of the model's total parameters** — worth knowing if you ever want to shrink this
  model, since this one layer is where almost all its size comes from.
- `Dropout(0.1)` randomly zeroes out 10% of that layer's outputs on every training step
  (a different random 10% each time), forcing the network to not rely too heavily on any
  single unit. This is the second of the two changes — alongside `BatchNormalization` —
  that fixed the overfitting the walkthrough describes.
- The same `Dense(64) + Dropout(0.1)` pattern repeats once more, narrowing 128 features
  down to 64 before the final output layer.

In [ ]:
# Output Layer for Binary Classification
model.add(Dense(1, activation='sigmoid'))

model.summary()

**What this cell does.** The final layer: a single unit with a **sigmoid**
activation, which squashes its output into the range $(0, 1)$ — interpretable as "how
much does this look like a dog", where values near 1 mean dog and values near 0 mean cat
(matching the label convention from Section 4: cats=0, dogs=1). One output unit is all a
*binary* classifier needs; there's no reason to use two units with softmax here the way
you would for 3+ classes.

`model.summary()` prints a table of every layer, its output shape, and its parameter
count. Working through the same conv/pool formulas used above by hand gives the numbers
you should see printed:

| layer | output shape | parameters |
|---|---|---|
| Conv2D (32 filters) | (254, 254, 32) | 896 |
| BatchNormalization | (254, 254, 32) | 128 |
| MaxPooling2D | (127, 127, 32) | 0 |
| Conv2D (64 filters) | (125, 125, 64) | 18,496 |
| BatchNormalization | (125, 125, 64) | 256 |
| MaxPooling2D | (62, 62, 64) | 0 |
| Conv2D (128 filters) | (60, 60, 128) | 73,856 |
| BatchNormalization | (60, 60, 128) | 512 |
| MaxPooling2D | (30, 30, 128) | 0 |
| Flatten | (115200,) | 0 |
| Dense (128) | (128,) | 14,745,728 |
| Dropout | (128,) | 0 |
| Dense (64) | (64,) | 8,256 |
| Dropout | (64,) | 0 |
| Dense (1, sigmoid) | (1,) | 65 |
| **Total** | | **14,848,193** (14,847,745 trainable + 448 non-trainable) |

The 448 non-trainable parameters are the three `BatchNormalization` layers' running mean
and variance — tracked during training but not updated by the optimizer directly.
(These figures are computed by hand from the standard convolution/pooling/dense parameter
formulas, since this notebook was written in an environment without TensorFlow installed
to run `model.summary()` directly — they should match what Colab prints exactly; if they
don't, double check `IMG_SIZE` in Section 4 is really `(256, 256)`.)

In [ ]:
# ==========================================
# 5. MODEL COMPILATION & TRAINING
# ==========================================
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Execute training across 10 epochs
history = model.fit(train_ds, epochs=10, validation_data=validation_ds)

**What this cell does.**

- `model.compile(...)` configures how the model will learn: `optimizer='adam'` (see
  [adam.md](adam.md) in this series if you want the mechanics — in short, an adaptive
  per-parameter learning rate that's a strong default for most problems);
  `loss='binary_crossentropy'`, the standard loss for a two-class sigmoid output; and
  `metrics=['accuracy']`, which just adds accuracy to what gets tracked and printed
  every epoch (the loss is what's actually optimized).
- `model.fit(train_ds, epochs=10, validation_data=validation_ds)` runs training: 10 full
  passes over `train_ds`, and after each one, evaluates on `validation_ds` — data the
  model never trains on — to check whether it's actually generalizing or just memorizing
  the training set. The returned `history` object records the loss and accuracy (both
  training and validation) at the end of every epoch, which is exactly what the next
  cell plots.
- Expect this cell to take a while: with a Colab GPU, a few minutes per epoch depending
  on dataset size; on CPU, considerably longer. This is also the step that can't be run
  in the environment this notebook was written in, since it needs the actual downloaded
  dataset from Section 2.

In [ ]:
# ==========================================
# 6. PLOT TRAINING HISTORY
# ==========================================
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', color='blue')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='red')
plt.title('Accuracy vs. Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='red')
plt.title('Loss vs. Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

**What this cell does.** Plots both curves side by side, which is the standard way
to read whether a model overfit:

- **Left (accuracy):** if the blue (training) line keeps climbing while the red
  (validation) line flattens out or dips, that gap *is* overfitting — the model is
  getting better at the training images specifically, not at the underlying task. With
  `BatchNormalization` and `Dropout` in place, expect these two lines to track each other
  much more closely than they would without those layers.
- **Right (loss):** the same idea in loss terms — watch for validation loss flattening
  or starting to *increase* even while training loss keeps falling. That upturn is
  usually the clearest single sign of overfitting, often visible before it shows up in
  the accuracy curve.

`history.history` is just a Python dict; `history.history['accuracy']` is the list of
training-accuracy values, one per epoch, and the other three keys follow the same
pattern.

In [ ]:
# ==========================================
# 7. PREDICT ON A NEW IMAGE
# ==========================================
def predict_image(path):
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.imshow(img_rgb)
    plt.axis('off')
    plt.show()

    img_resized = cv2.resize(img_rgb, (256, 256))
    img_array = img_resized.reshape((1, 256, 256, 3)) / 255.0

    prediction = model.predict(img_array)[0][0]

    label = "Dog" if prediction > 0.5 else "Cat"
    confidence = prediction if prediction > 0.5 else 1 - prediction
    print(f"Prediction: {label}  (confidence: {confidence:.2%})")

# Example usage -- replace with the path to your own test image
predict_image('/content/dog_test.jpg')

**What this cell does.** Runs the trained model on a single new photo, end to end:

1. `cv2.imread(path)` loads the image; OpenCV loads color images in **BGR** channel
   order, but the training pipeline in Section 4 (`image_dataset_from_directory`) decodes
   images as **RGB**. That mismatch matters here, not just cosmetically: feeding the
   model a BGR array when it was trained on RGB arrays would quietly hurt accuracy (red
   and blue channels swapped is a real, different-looking input to a CNN, even though a
   human glancing at a *mis-displayed* BGR-as-RGB plot might not immediately notice). So
   `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` converts once, right after loading, and
   `img_rgb` is what's used for **both** the preview plot and the actual prediction
   below — not just the plot.
2. `cv2.resize(img_rgb, (256, 256))` matches the size every training image was resized
   to — the model's first layer has a fixed `input_shape=(256,256,3)` and cannot accept
   anything else.
3. `.reshape((1, 256, 256, 3))` adds a batch dimension of size 1, since `model.predict`
   always expects a *batch* of images, even a batch of one; dividing by 255 applies the
   exact same normalization used in Section 5.
4. `model.predict(img_array)[0][0]` runs the forward pass and pulls out the single
   sigmoid value. Since Section 4 confirmed `cats → 0`, `dogs → 1`, a value above 0.5
   means the model leans "dog"; the `confidence` line just reports how far from 0.5 the
   prediction landed, as a percentage.

Replace `'/content/dog_test.jpg'` with the path to any photo you've uploaded to the Colab
session (drag a file into the Colab file browser on the left, or use
`files.upload()` again as in Section 2) to try it on your own images.